In [1]:
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
import pandas as pd
from langchain_chroma import Chroma
import torch
# print(torch.__version__, torch.version.cuda)
# print(torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [2]:
#importing dataset

df = pd.read_csv("../datasets/pubmed abstracts/pubmed_abstracts copy.csv")
df.head()

,Unnamed: 0,deep_learning
0,0,(['Magnetic resonance spectroscopic imaging (M...
1,1,(['Existing deep convolutional neural networks...
2,2,(['Deep learning techniques have been increasi...
3,3,(['The original article unfortunately containe...
4,4,(['The most common applications of artificial ...


In [3]:
#loading dataset in langchain

docs = []
for col in df.columns:
    if col == "Unnamed: 0":
        continue
    text = "\n".join(df[col].dropna().astype(str))
    docs.append(Document(page_content=text, metadata={"column": col}))

print(len(docs))   # = number of columns
print(docs[0].metadata)
print(type(docs[0].page_content))

1
{'column': 'deep_learning'}
<class 'str'>


In [4]:
#Spliting text

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
texts = text_splitter.split_documents(docs)
print(len(texts)) 
print(texts[5].page_content)

22842
by the classification branch. Feeding a chest radiograph to the trained Thorax-Net, a diagnosis is obtained by averaging and binarizing the outputs of two branches. The proposed Thorax-Net model has been evaluated against three state-of-the-art deep learning models using the patientwise official split of the ChestX-ray14 dataset and against other five deep learning models using the imagewise random data split. Our results show that Thorax-Net achieves an average per-class area under the receiver operating characteristic curve (AUC) of 0.7876 and 0.896 in both experiments, respectively, which are higher than the AUC values obtained by other deep models when they were all trained with no external data.'], 'Thorax-Net: An Attention Regularized Deep Neural Network for Classification of Thoracic Diseases on Chest Radiography.')


In [35]:
#Embeddings Model 

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"batch_size": 512},
    )

embeddings._client.half()                 # fp16
embeddings._client.max_seq_length = 256   # truncate long abstracts

query_embedding = embeddings.embed_query("What is a sentence embedding?")
doc_embeddings = embeddings.embed_documents(
    [
        "Sentence embeddings map text to dense vectors.",
        "LangChain provides a standard Embeddings interface.",
    ]
)

print(type(embeddings))
print(hasattr(embeddings, "_client"), hasattr(embeddings, "client"))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 695.76it/s]


<class 'langchain_huggingface.embeddings.huggingface.HuggingFaceEmbeddings'>
True False


In [7]:
m = embeddings._client
print(m.max_seq_length)                  # 256
print(next(m.parameters()).dtype)        # torch.float16
print(next(m.parameters()).device)       # cuda:0

256
torch.float16
cuda:0


In [4]:
#Embeddigns & Vector storage

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

vector_store.add_documents(texts)

NameError: name 'texts' is not defined

In [ ]:
#Timeline of storage

from tqdm import tqdm

max_bs = vector_store._client.get_max_batch_size()   # 5461 for you
BATCH = min(2000, max_bs)                            # stay safely under it

for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i + BATCH]
    ids = [str(d.metadata.get("pmid", i + j)) for j, d in enumerate(batch)]
    vector_store.add_documents(batch, ids=ids)

100%|██████████| 12/12 [1:02:40<00:00, 313.33s/it]


In [5]:
#Retrieval

vector_store = Chroma(
    collection_name="example_collection",
    embedding_function=embeddings,
    persist_directory="chroma_langchain_db",
)

retriever = vector_store.as_retriever()

retreived = retriever.invoke("what is adhd?")
print(len(retreived))
print(retreived)

4
[Document(id='22834', metadata={'column': 'deep_learning'}, page_content='(["Studies on ADHD in educational settings indicate that a student\'s motivation for learning is significantly related with the student\'s grade point average. The present study examined the relationship between ADHD symptoms and student academic achievement by considering the student\'s approach to and motivation for learning."], \'Study Habits, Motives, and Strategies of College Students With Symptoms of ADHD.\')'), Document(id='18436', metadata={'column': 'deep_learning'}, page_content="ADHD diagnosed participants showed higher differences as compared to those of normally developing participants, those differences were only statistically significant for medium intensity movements. On the other hand, the medicated ADHD participants showed statistically different behavior for low intensity movements.'], 'Using Recurrent Neural Networks to Compare Movement Patterns in ADHD and Normally Developing Children Based

In [29]:
#Setting up LLM

llm = ChatOllama(
    model="gemma4:e4b", 
    temperature = 0.5
)

In [64]:
#Query LLM

query = "what is adhd"
# response = llm.invoke(query)
# print(response.text)

# print(type(response))
# print(type(response.text))
print(type(response))

<class 'str'>


In [50]:
import ollama
r = ollama.chat(model="gemma4:e4b", messages=[{"role": "user", "content": "Say hi in JSON: {\"msg\": ...}"}])
print(repr(r.message.content))

'```json\n{"msg": "Hi there!"}\n```'


In [65]:
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import HallucinationMetric, FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.models import OllamaModel

llm = OllamaModel(
    model="gemma4:e4b", 
    base_url="http://localhost:11434",
    temperature = 0.5
    )


passage = [
    "Attention Deficit Hyperactivity Disorder (ADHD) is one of the most prevalent neuropsychiatric disorders in childhood and adolescence and its diagnosis is based on clinical interviews, symptom questionnaires, and neuropsychological testing. Much research effort has been undertaken to evaluate the usefulness of neurophysiological (EEG) data to aid this diagnostic process. In the current study, we applied deep learning methods on event-related EEG data to examine whether it is possible to distinguish ADHD patients from healthy controls using purely neurophysiological measures. The same was done to distinguish between ADHD subtypes. The results show that the applied deep learning model (\"EEGNet\") was able to distinguish between both ADHD subtypes and healthy controls with an accuracy of up to 83%. However, a significant fraction of individuals could not be classified correctly. It is shown that neurophysiological processes indicating attentional selection associated with superior parietal cortical areas were the most important for that. Using the applied deep learning method, it was not possible to distinguish ADHD subtypes from each other. This is the first study showing that deep learning methods applied to EEG data are able to dissociate between ADHD patients and healthy controls. The results show that the applied method reflects a promising means to support clinical diagnosis in ADHD. However, more work needs to be done to increase the reliability of the taken approach.Deep Learning Based on Event-Related EEG Differentiates Children with ADHD from Healthy Controls.",
    "Attention deficit/hyperactivity disorder(ADHD) is a behavioral disorder syndrome found mainly in school-age population. At present, the diagnosis of ADHD mainly depends on the subjective methods, leading to the high rate of misdiagnosis and missed-diagnosis. To solve these problems, we proposed an algorithm for classifying ADHD objectively based on convolutional neural network. At first, preprocessing steps, including skull stripping, Gaussian kernel smoothing, et al., were applied to brain magnetic resonance imaging(MRI). Then, coarse segmentation was used for selecting the right caudate nucleus, left precuneus, and left superior frontal gyrus region. Finally, a 3 level convolutional neural network was used for classification. Experimental results showed that the proposed algorithm was capable of classifying ADHD and normal groups effectively, the classification accuracies obtained by the right caudate nucleus and the left precuneus brain regions were greater than the highest classification accuracy(62.52%) in the ADHD-200 competition, and among 3 brain regions in ADHD and the normal groups, the classification accuracy from the right caudate nucleus was the highest. It is well concluded that the method for classification of ADHD and normal groups proposed in this paper utilizing the coarse segmentation and deep learning is a useful method for the purpose. The classification accuracy of the proposed method is high, and the calculation is simple. And the method is able to extract the unobvious image features better, and can overcome the shortcomings of traditional methods of MRI brain area segmentation, which are time-consuming and highly complicate. The method provides an objective diagnosis approach for ADHD.Study of attention deficit/hyperactivity disorder classification based on convolutional neural networks",
    "Brain functional imaging data, especially functional magnetic resonance imaging (fMRI) data, have been employed to reflect functional integration of the brain. Alteration in brain functional connectivity (FC) is expected to provide potential biomarkers for classifying or predicting brain disorders. In this paper, we present a comprehensive review in order to provide guidance about the available brain FC measures and typical classification strategies. We survey the state-of-the-art FC analysis methods including widely used static functional connectivity (SFC) and more recently proposed dynamic functional connectivity (DFC). Temporal correlations among regions of interest (ROIs), data-driven spatial network and functional network connectivity (FNC) are often computed to reflect SFC from different angles. SFC can be extended to DFC using a sliding-window framework, and intrinsic connectivity states along the time-varying connectivity patterns are typically extracted using clustering or decomposition approaches. We also briefly summarize window-less DFC approaches. Subsequently, we highlight various strategies for feature selection including the filter, wrapper and embedded methods. In terms of model building, we include traditional classifiers as well as more recently applied deep learning methods. Moreover, we review representative applications with remarkable classification accuracy for psychosis and mood disorders, neurodevelopmental disorder, and neurological disorders using fMRI data. Schizophrenia, bipolar disorder, autism spectrum disorder (ASD), attention deficit hyperactivity disorder (ADHD), Alzheimer's disease and mild cognitive impairment (MCI) are discussed. Finally, challenges in the field are pointed out with respect to the inaccurate diagnosis labeling, the abundant number of possible features and the difficulty in validation. Some suggestions for future work are also provided.Classification and Prediction of Brain Disorders Using Functional Connectivity: Promising but Challenging.",
    "Studies on ADHD in educational settings indicate that a student's motivation for learning is significantly related with the student's grade point average. The present study examined the relationship between ADHD symptoms and student academic achievement by considering the student's approach to and motivation for learning.Study Habits, Motives, and Strategies of College Students With Symptoms of ADHD.",
    "The health risks to babies from pollutants in house dust may be 100 times greater than for adults. The young ingest more dust and are up to ten times more vulnerable to such exposures. House dust is the main exposure source for infants to allergens, lead, and PBDEs, as well as a major source of exposure to pesticides, PAHs, Gram-negative bacteria, arsenic, cadmium, chromium, phthalates, phenols, and other EDCs, mutagens, and carcinogens. Median or upper percentile concentrations in house dust of lead and several pesticides and PAHs may exceed health-based standards in North America. Early contact with pollutants among the very young is associated with higher rates of chronic illness such as asthma, loss of intelligence, ADHD, and cancer in children and adults. The potential of infants, who live in areas with soil contaminated by automotive and industrial emissions, can be given more protection by improved home cleaning and hand washing. Babies who live in houses built before 1978 have a prospective need for protection against lead exposures; homes built before 1940 have even higher lead exposure risks. The concentration of pollutants in house dust may be 2-32 times higher than that found in the soil near a house. Reducing infant exposures, at this critical time in their development, may reduce lifetime health costs, improve early learning, and increase adult productivity. Some interventions show a very rapid payback. Two large studies provide evidence that home visits to reduce the exposure of children with poorly controlled asthma triggers may return more than 100% on investment in 1 yr in reduced health costs. The tools provided to families during home visits, designed to reduce dust exposures, included vacuum cleaners with dirt finders and HEPA filtration, allergy control bedding covers, high-quality door mats, and HEPA air filters. Infants receive their highest exposure to pollutants in dust at home, where they spend the most time, and where the family has the most mitigation control. Normal vacuum cleaning allows deep dust to build up in carpets where it can be brought to the surface and become airborne as a result of activity on the carpet. Vacuums with dirt finders allow families to use the three-spot test to monitor deep dust, which can reinforce good cleaning habits. Motivated families that receive home visits from trained outreach workers can monitor and reduce dust exposures by 90% or more in 1 wk. The cost of such visits is low considering the reduction of risks achieved. Improved home cleaning is one of the first results observed among families who receive home visits from MHEs and CHWs. We believe that proven intervention methods can reduce the exposure of infants to pollutants in house dust, while recognizing that much remains to be learned about improving the effectiveness of such methods. Monitoring and reducing exposure of infants to pollutants in house dust.",
    "Neuroimaging-based single subject prediction of brain disorders has gained increasing attention in recent years. Using a variety of neuroimaging modalities such as structural, functional and diffusion MRI, along with machine learning techniques, hundreds of studies have been carried out for accurate classification of patients with heterogeneous mental and neurodegenerative disorders such as schizophrenia and Alzheimer's disease. More than 500 studies have been published during the past quarter century on single subject prediction focused on a multiple brain disorders. In the first part of this study, we provide a survey of more than 200 reports in this field with a focus on schizophrenia, mild cognitive impairment (MCI), Alzheimer's disease (AD), depressive disorders, autism spectrum disease (ASD) and attention-deficit hyperactivity disorder (ADHD). Detailed information about those studies such as sample size, type and number of extracted features and reported accuracy are summarized and discussed. To our knowledge, this is by far the most comprehensive review of neuroimaging-based single subject prediction of brain disorders. In the second part, we present our opinion on major pitfalls of those studies from a machine learning point of view. Common biases are discussed and suggestions are provided. Moreover, emerging trends such as decentralized data sharing, multimodal brain imaging, differential diagnosis, disease subtype classification and deep learning are also discussed. Based on this survey, there is extensive evidence showing the great potential of neuroimaging data for single subject prediction of various disorders. However, the main bottleneck of this exciting field is still the limited sample size, which could be potentially addressed by modern data sharing models such as the ones discussed in this paper. Emerging big data technologies and advanced data-intensive machine learning methodologies such as deep learning have coincided with an increasing need for accurate, robust and generalizable single subject prediction of brain disorders during an exciting time. In this report, we survey the past and offer some opinions regarding the road ahead. Single subject prediction of brain disorders in neuroimaging: Promises and pitfalls.",
]

# --------------------------------------------------------
# 1. TEST CASE BEFORE RAG (Baseline Chatbot)
# --------------------------------------------------------
# No context is provided here because RAG isn't active yet.
test_case_before_rag = LLMTestCase(
    input=query,
    actual_output=response,
    context=passage,

)

# Use HallucinationMetric for standalone LLMs (requires a context list to check alignment)
# Note: For strict 'before' testing without context, you can provide the ground truth instead.
metric = HallucinationMetric(model=llm, threshold=0.5, include_reason=True) 
metric.measure(test_case_before_rag)

# # --------------------------------------------------------
# # 2. TEST CASE AFTER RAG (With Context)
# # --------------------------------------------------------
# test_case_after_rag = LLMTestCase(
#     input="What is the return policy for clearance items?",
#     actual_output="Clearance items are final sale and cannot be returned or exchanged.",
#     retrieval_context=[
#         "Section 4.2: All items marked as 'Clearance' or 'Final Sale' are ineligible for returns, refunds, or exchanges."
#     ]
# )

# # Use Faithfulness to ensure the answer strictly matches the retrieved context
# after_faithfulness = FaithfulnessMetric(threshold=0.7)
# after_relevance = AnswerRelevancyMetric(threshold=0.7)


# --------------------------------------------------------
# 3. RUN THE EVALUATIONS
# --------------------------------------------------------
print("Evaluating Baseline (Before RAG)...")
print(metric.score)    # 0 to 1
print(metric.reason)   # judge's explanation
print(metric.is_successful())
# evaluate([test_case_before_rag], [before_metric])

# print("\nEvaluating RAG Pipeline (After RAG)...")
# evaluate([test_case_after_rag], [after_faithfulness, after_relevance])


s:\Projects\rag-dissertation\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

ValidationError: 1 validation error for Verdicts
  Invalid JSON: EOF while parsing a value at line 1 column 0 [type=json_invalid, input_value='', input_type=str]
    For further information visit https://errors.pydantic.dev/2.14/v/json_invalid